# YOLOv10n NSFW Detector

In [1]:
# Install Ultralytics + a PINNED export toolchain.
#
# Root cause of the previous TFLite export crash:
#   Ultralytics auto-installs onnx2tf 1.28.x, which mis-converts YOLOv10's
#   end-to-end NMS head -> ONNX->TF dies on 'model.23/Concat' with a dim
#   mismatch ([1,4,300] vs [1,300,1]). The documented working ceiling is
#   onnx2tf <= 1.26.3. The auto-install also downgrades TF mid-run, but that
#   never takes effect because TF is already imported (needs a runtime restart).
#
# Fix: pin onnx2tf==1.26.3 and the TF stack HERE, before TensorFlow is imported,
# so the conversion runs on the correct, compatible versions in one clean run.
!pip install -q ultralytics==8.4.60 onnx2tf==1.26.3 tensorflow==2.19.1 tf_keras==2.19.0 "onnx<2.0" onnxslim onnxruntime onnx_graphsurgeon sng4onnx "ai-edge-litert>=1.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.6/146.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.6/445.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 106.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import os
import yaml
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import tensorflow as tf
from pathlib import Path
import shutil
import json
from datetime import datetime
import zipfile

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
class ScreenContentDetector:
    def __init__(self, dataset_path, model_name='yolov10n'):
        """Initialize the screen content detector."""
        self.dataset_path = Path(dataset_path)
        self.model_name = model_name
        self.data_yaml = self.dataset_path / 'data.yaml'
        self.output_dir = Path('output')
        self.output_dir.mkdir(exist_ok=True)

    def prepare_dataset(self, classes=None):
      """Locate the dataset's data.yaml and normalize it.

      Handles zips that wrap everything in a sub-folder (e.g. the archive
      extracts to <root>/combined_dataset/{train,val,data.yaml}): searches for
      data.yaml at any depth, re-roots the detector there, and rewrites the
      yaml's 'path' to an absolute dir so Ultralytics always finds the images.
      """
      yaml_path = self.dataset_path / 'data.yaml'
      if not yaml_path.exists():
          # search recursively; pick the shallowest match
          nested = sorted(self.dataset_path.rglob('data.yaml'), key=lambda q: len(q.parts))
          if nested:
              yaml_path = nested[0]
              print(f"Found nested data.yaml at: {yaml_path}")

      if yaml_path.exists():
          # point the dataset root at the folder that actually holds train/ val/
          self.dataset_path = yaml_path.parent
          with open(yaml_path, 'r') as f:
              data = yaml.safe_load(f) or {}
          # force an absolute, correct root regardless of how the yaml was authored
          data['path'] = str(self.dataset_path.absolute())
          if classes is not None and not data.get('names'):
              data['names'] = {i: name for i, name in enumerate(classes)}
          with open(yaml_path, 'w') as f:
              yaml.dump(data, f, default_flow_style=False, sort_keys=False)
          self.data_yaml = yaml_path
          print(f"Using data.yaml: {self.data_yaml}")
          print(f"  dataset root (path): {data['path']}")
          print(f"  train: {data.get('train')}  val: {data.get('val')}")
      else:
          # fallback: create a new one if none found anywhere
          print("No data.yaml found anywhere, creating a new one.")
          if classes is None:
              classes = ['safe', 'nudity', 'gore']

          data_config = {
              'path': str(self.dataset_path.absolute()),
              'train': 'train/images',
              'val': 'val/images',
              'names': {i: name for i, name in enumerate(classes)}
          }
          self.data_yaml = self.dataset_path / 'data.yaml'
          with open(self.data_yaml, 'w') as f:
              yaml.dump(data_config, f, default_flow_style=False)
          print(f"Created new data.yaml at: {self.data_yaml}")

      return self.data_yaml


    def train_model(self, epochs=50, imgsz=640, batch=16, weights='yolov10n.pt'):
        """Train YOLOv10n model. """

        print(f"Training model.")

        # Load pretrained YOLOv10n (COCO) weights for transfer learning
        model = YOLO(weights)

        # Select device from PyTorch (Colab GPU -> 0, else CPU)
        device = 0 if torch.cuda.is_available() else 'cpu'

        # Train the model
        results = model.train(
            data=str(self.data_yaml),
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=device,
            project='runs/train',
            name='screen_content_detector',
            patience=20,
            save=True,
            plots=True
        )

        # Resolve the actual run directory from the trainer instead of hardcoding
        # it -- Ultralytics nests runs/ differently across versions (8.3 vs 8.4).
        save_dir = Path(model.trainer.save_dir)
        self.best_model_path = save_dir / 'weights' / 'best.pt'
        print(f"Training complete! Best model saved at: {self.best_model_path}")

        return self.best_model_path

    def export_to_tflite(self, model_path=None):
        if model_path is None:
            model_path = self.best_model_path

        print(f"Export model")

        model = YOLO(model_path)

        # Export to float16 TFLite. The onnx2tf version that makes this work for
        # YOLOv10 (1.26.3) is pinned in the first cell. If the ONNX->TF step ever
        # fails again, fall back to ONNX so the run still produces a usable model
        # instead of crashing the whole pipeline.
        try:
            export_path = model.export(format='tflite', imgsz=640, half=True)
        except Exception as e:
            print(f"TFLite export failed: {e}")
            print("Falling back to ONNX export (onnx2tf-free).")
            onnx_path = model.export(format='onnx', imgsz=640, opset=12, simplify=True)
            onnx_output = self.output_dir / 'model.onnx'
            shutil.copy(onnx_path, onnx_output)
            print(f"ONNX model exported to: {onnx_output}")
            return onnx_output

        # Add metadata if possible (optional but strongly recommended)
        try:
            from tflite_support import metadata as _metadata
            labelmap_path = self.output_dir / 'labelmap.txt'
            populator = _metadata.MetadataPopulator.with_model_file(str(export_path))
            populator.load_associated_files([str(labelmap_path)])
            populator.populate()
            print("Metadata attached successfully.")
        except Exception as e:
            print("Metadata not added:", e)

        tflite_output = self.output_dir / 'model.tflite'
        shutil.copy(export_path, tflite_output)
        print(f"TFLite model exported to: {tflite_output}")

        return tflite_output


    def create_label_file(self):
        """Create labelmap.txt."""
        # Read classes from data.yaml
        with open(self.data_yaml, 'r') as f:
            data = yaml.safe_load(f)

        classes = data['names']

        # Create labelmap.txt
        labelmap_path = self.output_dir / 'labelmap.txt'
        with open(labelmap_path, 'w') as f:
            for idx in sorted(classes.keys()):
                f.write(f"{classes[idx]}\n")

        print(f"Label map created at: {labelmap_path}")
        print(f"Classes: {list(classes.values())}")

        return labelmap_path

    def run_inference_samples(self, model_path=None, num_samples=5):
        """Run inference."""
        if model_path is None:
            model_path = self.best_model_path

        print(f"Run inference.")

        # Load model
        model = YOLO(model_path)

        # Get sample images from validation set
        val_images_dir = self.dataset_path / 'val' / 'images'
        sample_images = list(val_images_dir.glob('*.jpg'))[:num_samples]
        sample_images += list(val_images_dir.glob('*.png'))[:num_samples - len(sample_images)]

        if not sample_images:
            print("No sample images found in validation set")
            return

        # Create inference output directory
        inference_dir = self.output_dir / 'inference_results'
        inference_dir.mkdir(exist_ok=True)

        results_summary = []

        for idx, img_path in enumerate(sample_images[:num_samples]):
            print(f"\nProcessing sample {idx + 1}/{num_samples}: {img_path.name}")

            # Run inference
            results = model(str(img_path), conf=0.25)

            # Get predictions
            boxes = results[0].boxes
            img = cv2.imread(str(img_path))

            if len(boxes) > 0:
                for box in boxes:
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    class_name = model.names[cls_id]

                    # Get bounding box coordinates
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Set color based on class
                    color_map = {
                        'safe': (0, 255, 0),      # Green
                        'nudity': (0, 0, 255),    # Red
                        'gore': (255, 0, 0)       # Blue
                    }
                    color = color_map.get(class_name, (255, 255, 255))

                    # Draw bounding box
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

                    # Add label
                    label = f"{class_name}: {conf:.2f}"
                    cv2.putText(img, label, (x1, y1 - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                    results_summary.append({
                        'image': img_path.name,
                        'class': class_name,
                        'confidence': conf
                    })

                    print(f"  Detected: {class_name} (confidence: {conf:.3f})")
            else:
                print(f"  No detections")
                results_summary.append({
                    'image': img_path.name,
                    'class': 'none',
                    'confidence': 0.0
                })

            # Save annotated image
            output_path = inference_dir / f"result_{idx + 1}_{img_path.name}"
            cv2.imwrite(str(output_path), img)
            print(f"  Saved to: {output_path}")

        # Print summary
        print("INFERENCE SUMMARY")
        for result in results_summary:
            print(f"{result['image']:30s} -> {result['class']:10s} ({result['confidence']:.3f})")

        return results_summary

In [4]:
class TFLiteInference:
    """TFLite inference class for mobile deployment."""
    def __init__(self, model_path, label_path):
        """ Initialize TFLite interpreter."""
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=str(model_path))
        self.interpreter.allocate_tensors()

        # Get input and output details
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

        # Load labels
        with open(label_path, 'r') as f:
            self.labels = [line.strip() for line in f.readlines()]

        print(f" TFLite model loaded")
        print(f"  Input shape: {self.input_details[0]['shape']}")
        print(f"  Labels: {self.labels}")

    def preprocess_image(self, image_path, target_size=640):
        """Preprocess image for TFLite inference."""
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (target_size, target_size))
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        return img

    def infer(self, image_path):
        """Run inference on an image."""
        # Preprocess
        input_data = self.preprocess_image(image_path)

        # Run inference
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        self.interpreter.invoke()

        # Get output
        output_data = self.interpreter.get_tensor(self.output_details[0]['index'])

        return output_data

In [5]:
from ultralytics import YOLO
from pathlib import Path
from datetime import datetime
import json

class ModelEvaluator:
    """Evaluate model and save metrics"""
    def __init__(self, model_path, data_yaml, output_dir='output/metrics_results'):
        self.model_path = Path(model_path)
        self.data_yaml = Path(data_yaml)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Load YOLO model
        self.model = YOLO(str(self.model_path))

    def evaluate(self, imgsz=640, batch=16, conf=0.25):
        """Run validation and compute metrics."""
        print("Evaluate model")

        results = self.model.val(
            data=str(self.data_yaml),
            imgsz=imgsz,
            batch=batch,
            conf=conf,
            split='val'
        )

        # Extract metrics from the Ultralytics DetMetrics object (results.box)
        box = results.box
        try:
            f1 = float(np.mean(box.f1)) if getattr(box, "f1", None) is not None and len(box.f1) else 0.0
        except Exception:
            f1 = 0.0
        metrics = {
            "precision": float(box.mp),       # mean precision across classes
            "recall": float(box.mr),          # mean recall across classes
            "f1_score": f1,
            "map50": float(box.map50),
            "map50_95": float(box.map),
        }

        # Timestamp for uniqueness
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = self.output_dir / f"eval_{timestamp}"
        run_dir.mkdir(exist_ok=True)

        # Save results as JSON and TXT
        json_path = run_dir / "metrics_summary.json"
        txt_path = run_dir / "metrics_summary.txt"

        with open(json_path, 'w') as jf:
            json.dump(metrics, jf, indent=4)

        with open(txt_path, 'w') as tfile:
            for k, v in metrics.items():
                tfile.write(f"{k}: {v:.4f}\n")

        print(f" Evaluation complete. Results saved at: {run_dir}")
        print("\nMetrics Summary:")
        for k, v in metrics.items():
            print(f"  {k:12s}: {v:.4f}")

        return metrics


In [6]:
def verify_extraction(dataset_path):
    """Check if dataset extracted correctly and contains required files."""
    required_dirs = ['train/images', 'train/labels', 'val/images', 'val/labels']
    missing = [d for d in required_dirs if not Path(dataset_path, d).exists()]
    if missing:
        print("⚠️ Missing folders after extraction:")
        for m in missing:
            print(f"  - {m}")
        for m in missing:
            Path(dataset_path, m).mkdir(parents=True, exist_ok=True)
        print("Created missing directories.")
    else:
        print("✅ All required folders found.")

    import glob
    img_files = glob.glob(f"{dataset_path}/train/images/*.*")
    if not img_files:
        print("❌ No images found in train/images.")
    else:
        print(f"✅ Found {len(img_files)} training images.")

In [7]:
def main():
    """Main training and export pipeline."""
    print("YOLOv10n")

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # --- Optional: load config.yaml if present (falls back to built-in defaults) ---
    cfg = {}
    if os.path.exists('config.yaml'):
        with open('config.yaml', 'r') as f:
            cfg = yaml.safe_load(f) or {}
        print("Loaded settings from config.yaml")
    else:
        print("config.yaml not found - using built-in defaults")

    ds_cfg    = cfg.get('dataset', {})
    train_cfg = cfg.get('train', {})
    eval_cfg  = cfg.get('eval', {})
    model_cfg = cfg.get('model', {})

    # Configuration
    DATASET_ZIP  = ds_cfg.get('zip_path', '/content/drive/MyDrive/Datasets/another_final_dataset.zip')
    DATASET_PATH = ds_cfg.get('extract_to', '/content/combined_dataset')
    CLASSES      = ds_cfg.get('classes', ['safe', 'nudity', 'gore'])
    WEIGHTS      = model_cfg.get('weights', 'yolov10n.pt')

    # Unzip dataset
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(DATASET_PATH)

    # --- Step 1: Initialize detector ---
    detector = ScreenContentDetector(
        dataset_path=DATASET_PATH,
        model_name='yolov10n'
    )

    # --- Step 2: Prepare dataset ---
    print("Prepare Dataset")
    detector.prepare_dataset(classes=CLASSES)

    # --- Step 3: Create label map (before export so TFLite metadata can attach) ---
    print("Create Label Map")
    labelmap_path = detector.create_label_file()

    # --- Step 4: Train model ---
    print("Train Model")
    model_path = detector.train_model(
        epochs=train_cfg.get('epochs', 50),
        imgsz=train_cfg.get('imgsz', 640),
        batch=train_cfg.get('batch', 16),
        weights=WEIGHTS,
    )

    # --- Step 5: Export to TFLite ---
    print("Export TFLite")
    tflite_path = detector.export_to_tflite(model_path)

    # --- Step 6: Run inference on samples ---
    print("Run Inference")
    detector.run_inference_samples(model_path, num_samples=5)

    # --- Step 7: Evaluate model ---
    print("Evaluate model")
    evaluator = ModelEvaluator(model_path, detector.data_yaml)
    metrics = evaluator.evaluate(imgsz=eval_cfg.get('imgsz', 640))

    # --- Step 8: Test TFLite inference (only if a .tflite was produced) ---
    if str(tflite_path).endswith('.tflite'):
        print("Test TFLite inference")
        tflite_model = TFLiteInference(tflite_path, labelmap_path)
    else:
        print(f"Skipping TFLite load test - no .tflite produced (got {tflite_path})")

    print("Complete")
    print("Output files:")
    print(f"  - TFLite model: {tflite_path}")
    print(f"  - Label map: {labelmap_path}")
    print(f"  - Inference results: output/inference_results/")


In [8]:
if __name__ == "__main__":
    main()

YOLOv10n
Mounted at /content/drive
Loaded settings from config.yaml
Prepare Dataset
Found nested data.yaml at: /content/combined_dataset/combined_dataset/combined_dataset/data.yaml
Using data.yaml: /content/combined_dataset/combined_dataset/combined_dataset/data.yaml
  dataset root (path): /content/combined_dataset/combined_dataset/combined_dataset
  train: train/images  val: val/images
Create Label Map
Label map created at: output/labelmap.txt
Classes: ['safe', 'nudity', 'gore']
Train Model
Training model.
New https://pypi.org/project/ultralytics/8.4.61 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/conten

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.0s, saved as '/content/runs/detect/runs/train/screen_content_detector/weights/best.onnx' (9.0 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 1.56s
Prepared 1 package in 8.01s
Installed 1 package in 63ms
 + onnxruntime-gpu==1.26.0

requirements: AutoUpdate success ✅ 10.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.1...
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 25.6files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.26.3...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



ERROR: The trace log is below.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnx2tf/utils/common_functions.py", line 312, in print_wrapper_func
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx2tf/utils/common_functions.py", line 385, in inverted_operation_enable_disable_wrapper_func
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx2tf/utils/common_functions.py", line 55, in get_replacement_parameter_wrapper_func
    func(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/onnx2tf/ops/Concat.py", line 282, in make_node
    tf.concat(
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/util/traceback_utils.py", line 153, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/layers/core/tf_op_layer.py", line

TypeError: object of type 'NoneType' has no len()

In [ ]:
#import shutil

# Path to the folder you want to delete
#folder_path = 'combined_dataset'

# Deleting the folder
#shutil.rmtree(folder_path)

In [11]:
from google.colab import files
import shutil
import os
import zipfile
from tqdm.notebook import tqdm  # progress bar in Colab

def zip_with_progress(src_paths, zip_name="/content/final.zip"):
    file_paths = []
    for src in src_paths:
        if os.path.isdir(src):
            # Walk through directories
            for root, _, files in os.walk(src):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, os.path.commonpath(src_paths))
                    file_paths.append((full_path, arcname))
        elif os.path.isfile(src):
            # Single file
            arcname = os.path.basename(src)
            file_paths.append((src, arcname))

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for full_path, arcname in tqdm(file_paths, desc="Zipping files", unit="file"):
            zipf.write(full_path, arcname)

    return zip_name


# Example usage:
src_items = [
    "/content/output",             # folder
    "/content/runs",                # another folder
    "/content/yolo11n.pt",          # individual file
    "/content/calibration_image_sample_data_20x128x128x3_float32.npy"
]

zip_path = zip_with_progress(src_items, "/content/final.zip")

# Download the zip
files.download(zip_path)


Zipping files:   0%|          | 0/27 [00:00<?, ?file/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import shutil
shutil.copytree("/content/runs/detect/runs/train/screen_content_detector",
                "/content/drive/MyDrive/THESIS_runs/screen_content_detector",
                dirs_exist_ok=True)
print("saved best.pt + best.onnx to Drive")


saved best.pt + best.onnx to Drive


In [10]:
from ultralytics import YOLO
m = YOLO("/content/drive/MyDrive/THESIS_runs/screen_content_detector/weights/best.pt")
m.model.model[-1].end2end = False          # <-- the fix: drop the baked-in NMS
p = m.export(format="tflite", imgsz=640, half=True)
print("EXPORTED:", p)


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv10n summary (fused): 126 layers, 2,695,586 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/THESIS_runs/screen_content_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (5.5 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.6s, saved as '/content/drive/MyDrive/THESIS_runs/screen_content_detector/weights/best.onnx' (9.0 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.1...
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.26.3...
Saved artifact at '/content/drive/MyDrive/THESIS_runs/screen_content_detector/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpe